In [2]:
import os
import json
import hashlib
import shutil
from collections import defaultdict
from genson import SchemaBuilder


In [5]:

def cluster_json_directory(input_dir="ai-json-files", output_dir="xyz_clustered"):
    if not os.path.exists(input_dir):
        print(f"Error: The directory '{input_dir}' does not exist.")
        return

    clusters = defaultdict(list)
    schema_map = {}

    print(f"Scanning '{input_dir}' for JSON files...")

    # 1. Parse files and generate schema fingerprints
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.json'):
            filepath = os.path.join(input_dir, filename)
            
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    
                    # Infer schema using genson
                    builder = SchemaBuilder()
                    builder.add_object(data)
                    schema_dict = builder.to_schema()
                    
                    # Create a deterministic string and hash for grouping
                    schema_string = json.dumps(schema_dict, sort_keys=True)
                    schema_hash = hashlib.md5(schema_string.encode('utf-8')).hexdigest()
                    
                    clusters[schema_hash].append(filepath)
                    if schema_hash not in schema_map:
                        schema_map[schema_hash] = schema_dict
            except (json.JSONDecodeError, IOError) as e:
                print(f"  ![Skipped] Failed to process {filename}: {e}")

    if not clusters:
        print("No valid JSON files found to cluster.")
        return

    # 2. Organize files into physical clusters
    print(f"\nFound {len(clusters)} unique schema structures. Organizing files...")
    
    for idx, (schema_hash, files) in enumerate(clusters.items(), start=1):
        # Create a specific folder for this cluster group
        short_hash = schema_hash[:8]
        group_folder = os.path.join(output_dir, f"schema_group_{idx}_{short_hash}")
        os.makedirs(group_folder, exist_ok=True)
        
        # Save the master schema rule for this group
        with open(os.path.join(group_folder, "detected_schema.json"), "w") as sf:
            json.dump(schema_map[schema_hash], sf, indent=2)
            
        # Copy the matching JSON files into this group's folder
        for file in files:
            shutil.copy(file, group_folder)
            
        print(f"  -> Group {idx} ({short_hash}): Packed {len(files)} files into {group_folder}")

    print(f"\nSuccess! Your clusters are ready inside the '{output_dir}' directory.")

if __name__ == "__main__":
    # Assumes 'xyz' is in the same folder you run this script from
    cluster_json_directory(input_dir="ai-json-files", output_dir="clustered_ai-json-files")

Scanning 'ai-json-files' for JSON files...
  ![Skipped] Failed to process ai_219.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_223.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_242.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_203.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_239.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_238.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_280.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_296.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_279.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_310.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped] Failed to process ai_214.json: Expecting value: line 1 column 1 (char 0)
  ![Skipped

In [18]:
import os
import fnmatch

def count_matching_files_in_tree(root_dir, pattern="ai_*.json"):
    """
    Traverses a nested directory structure and counts only the files
    that match a specific naming pattern (e.g., 'ai_*.json').
    
    :param root_dir: Path to the target root directory
    :param pattern: Shell-style wildcard pattern to match filenames against
    :return: A dictionary with directory paths as keys and matching file counts as values
    """
    if not os.path.exists(root_dir):
        print(f"Error: The directory '{root_dir}' does not exist.")
        return {}

    directory_counts = {}

    print(f"Counting files matching '{pattern}' starting from: {root_dir}\n")
    
    for dirpath, dirnames, filenames in os.walk(root_dir):
        # Filter the filenames list to only include matches to your pattern
        matching_files = fnmatch.filter(filenames, pattern)
        
        # Only record directories that actually contain at least one matching file
        if matching_files:
            directory_counts[dirpath] = len(matching_files)
        else:
            # Optional: Include directories with 0 matches if you want a complete map
            directory_counts[dirpath] = 0

    return directory_counts

# --- Example Usage ---
if __name__ == "__main__":
    target_folder = "clustered_ai-json-files"  # Change this to your actual target folder
    
    # Run the function with your specific naming criteria
    counts = count_matching_files_in_tree(target_folder, pattern="ai_*.json")
    
    # Display the results neatly
    print(f"{'Directory Path':<50} | {'Matching Files':<10}")
    print("-" * 65)
    count_json_schema = 0

    for folder, count in counts.items():
        # Optional: Skip printing folders that have 0 matching files to keep output clean
        if count > 1:
            print(f"{folder:<50} | {count:<10}")
            count_json_schema = count_json_schema + 1
    print(f"\nTotal JSON schema groups found: {count_json_schema}")

Counting files matching 'ai_*.json' starting from: clustered_ai-json-files

Directory Path                                     | Matching Files
-----------------------------------------------------------------
clustered_ai-json-files/schema_group_115_209e3ed5  | 3         
clustered_ai-json-files/schema_group_41_a7be8828   | 3         
clustered_ai-json-files/schema_group_7_c9b8a033    | 5         
clustered_ai-json-files/schema_group_105_a71434ec  | 6         
clustered_ai-json-files/schema_group_63_cdfd3a3c   | 2         
clustered_ai-json-files/schema_group_25_e21682d1   | 61        
clustered_ai-json-files/schema_group_15_87f9c798   | 7         
clustered_ai-json-files/schema_group_157_3258a372  | 2         
clustered_ai-json-files/schema_group_175_9844d284  | 2         
clustered_ai-json-files/schema_group_6_3a0f96f1    | 9         
clustered_ai-json-files/schema_group_33_d25a6ed6   | 9         
clustered_ai-json-files/schema_group_34_b5c93539   | 4         
clustered_ai-json-file

In [19]:
def generate_unified_schema(input_dir="processed_schema_workspace", pattern="ai_*.json", output_file="master_schema.json"):
    """Scans a directory tree for specific JSON files and merges them into a single master schema.

    This function utilizes the 'genson' library to iteratively ingest multiple 
    JSON structures. It handles schema drift by dynamically identifying optional 
    fields, varying array structures, and multi-type values across all processed files.

    Args:
        input_dir (str): The root directory path to start searching for files. 
            Defaults to "processed_schema_workspace".
        pattern (str): A Unix shell-style wildcard pattern to filter files by name. 
            Defaults to "ai_*.json".
        output_file (str): The file path where the final, compiled JSON schema 
            will be saved. Defaults to "master_schema.json".

    Returns:
        None: Writes the output directly to a file and prints status updates to the console.
    """
    # Quick sanity check: Ensure the user provided a directory that actually exists
    if not os.path.exists(input_dir):
        print(f"Error: The directory '{input_dir}' does not exist.")
        return

    # Initialize SchemaBuilder. A single instance is maintained across all loops
    # so that it can continually absorb and merge new data structures.
    master_builder = SchemaBuilder()
    file_count = 0

    print(f"Scanning '{input_dir}' for files matching '{pattern}'...")

    # os.walk travels down through any nested folder structures recursively
    for dirpath, _, filenames in os.walk(input_dir):
        # Filter the file list in the current directory against our pattern (e.g., ai_*.json)
        for filename in fnmatch.filter(filenames, pattern):
            filepath = os.path.join(dirpath, filename)
            
            try:
                # Open and parse the JSON file safely using UTF-8 encoding
                with open(filepath, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    
                    # Feed the parsed JSON object into the single master builder instance.
                    # Genson automatically calculates differences (e.g., marks missing keys 
                    # as optional, updates polymorphic data types like int vs string)
                    master_builder.add_object(data)
                    file_count += 1
                    
            except json.JSONDecodeError as e:
                print(f"  ![JSON Error] Skipped {filename} due to malformed syntax: {e}")
            except IOError as e:
                print(f"  ![File Error] Could not read/open {filename}: {e}")

    # Guard clause: If no files matched our criteria, we stop here to avoid creating an empty schema
    if file_count == 0:
        print("No matching JSON files found to generate a schema from.")
        return

    # Convert the internal Genson builder accumulator state into a standard python dictionary schema
    master_schema = master_builder.to_schema()

    # Write the resulting unified dictionary layout into a beautifully indented text file
    with open(output_file, 'w', encoding='utf-8') as sf:
        json.dump(master_schema, sf, indent=2)

    print(f"\nSuccess! Successfully swallowed {file_count} files into a single schema.")
    print(f"Your master schema has been saved to: '{output_file}'")

if __name__ == "__main__":
    # Execution point: Assumes a folder named 'processed_schema_workspace' sits in the same directory where you run this script
    generate_unified_schema(input_dir="processed_schema_workspace", pattern="ai_*.json", output_file="master_schema.json")

Scanning 'processed_schema_workspace' for files matching 'ai_*.json'...
  ![JSON Error] Skipped ai_219.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_223.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_242.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_203.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_239.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_238.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_280.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_296.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
  ![JSON Error] Skipped ai_279.json due to malformed syntax: Expecting value: line 1 column 1 (char 0)
 